# Phase 3 -- Full LoRA training (Sessions 1-3 train / Session 4 validation)

Trains ONE adapter per run (`ADAPTER_VARIANT = "nohist"` or `"hist3"`, set below). Run
this notebook twice, as two separate Kaggle sessions/copies, once per variant -- their
checkpoints/output directories are kept completely separate.

Implements the empirically-validated architecture from the Phase-3 smoke test
(`kaggle_phase3_smoketest.ipynb`, cleared 7/7 gates):
  - training forward: `model.thinker(**batch)` (top-level `Qwen2_5OmniForConditionalGeneration`
    has no usable forward -- proved by Diagnostics D/E, not assumed here)
  - inference: top-level `model.generate(...)`
  - 144 exact `thinker.model.layers.{0..35}.self_attn.{q,k,v,o}_proj` LoRA targets,
    7,372,800 trainable parameters, audio/vision/talker frozen, fp16, T4x2

Set `SMOKE_MODE = True` first and Run All to verify a few optimizer steps + a tiny
validation generation pass on 24 train / 24 val examples. Only after that finishes
cleanly, set `SMOKE_MODE = False` and Restart & Run All for the real run.

Session 5 is never read, written, or referenced anywhere in this notebook.


In [1]:
# ============================ CONFIG ============================
MODEL_ID = "Qwen/Qwen2.5-Omni-3B"
SEED = 11

# ---- set these two by hand for each of the two required runs ----
ADAPTER_VARIANT = "hist3"   # "nohist" or "hist3" -- selects which format this run trains+validates on
SMOKE_MODE = False            # True: tiny data, 1 epoch, patience=1, separate output dir. False: full run.

# ---- frozen protocol hyperparameters (Phase-3, approved) ----
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
ADAMW_LR = 1e-4
ADAMW_BETAS = (0.9, 0.95)
# ADAMW default weight_decay=0.01 is used (not explicitly specified in the frozen list;
# kept as PyTorch's own AdamW default rather than silently changed to 0 -- flagged in the
# writeup as a precommitted default, not a controlled-comparison claim).
MICROBATCH = 1
GRAD_ACCUM = 8                 # effective batch size = 8
MAX_EPOCHS_FULL = 8
PATIENCE_FULL = 3
MAX_EPOCHS_SMOKE = 1
PATIENCE_SMOKE = 1
CHECKPOINT_METRIC = "macro_f1"
MACRO_F1_ROUND_NDIGITS = 4      # tie-break precision: 4 dp in [0,1] scale == 2 dp as a percentage,
                                 # matching this project's established metric-reporting precision
SAVE_EVERY_N_OPT_STEPS = 100     # mid-epoch checkpoint cadence, in OPTIMIZER steps (only ever
                                 # checkpointed right after optimizer.step(), never mid-accumulation)
MAX_NEW_TOKENS = 12             # identical generation setting to every prior notebook in this project

assert ADAPTER_VARIANT in ("nohist", "hist3")

RUN_TAG = f"{ADAPTER_VARIANT}_{'smoke' if SMOKE_MODE else 'full'}"
OUTPUT_DIR = f"/kaggle/working/phase3_train_{ADAPTER_VARIANT}_{'smoke' if SMOKE_MODE else 'full'}"
# Optional: point this at a previous COMMITTED run's output (added as a Kaggle Input dataset)
# to resume training across a fresh Kaggle session, e.g. "/kaggle/input/phase3-nohist-ckpt-v1".
# Leave as None for a same-session resume (checkpoints already in OUTPUT_DIR) or a fresh start.
RESUME_INPUT_DIR = None

EVAL_LIB_SHA256_EXPECTED = "24fac1f49d6f049c1485b4769ce523a292ae692fad4657a9fab7be6af6034298"

print("ADAPTER_VARIANT:", ADAPTER_VARIANT, "| SMOKE_MODE:", SMOKE_MODE)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("SCOPE: full LoRA training for ONE adapter variant. No Session 5.")


ADAPTER_VARIANT: hist3 | SMOKE_MODE: False
OUTPUT_DIR: /kaggle/working/phase3_train_hist3_full
SCOPE: full LoRA training for ONE adapter variant. No Session 5.


In [2]:
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
pip("-U", "transformers>=4.52.3", "accelerate>=0.34", "qwen-omni-utils", "librosa", "soundfile", "peft>=0.11.1")
# NOTE: deliberately NOT reinstalling numpy/scipy (see kaggle_phase3_smoketest.ipynb history --
# force-reinstalling them to "latest" produced an internally-inconsistent numpy install on this
# image). Kaggle's own pre-installed numpy/scipy pairing is used as-is.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "scikit-learn"], check=True)
import numpy as _np_check, scipy as _sp_check
print("numpy  :", _np_check.__version__, "->", _np_check.__file__)
print("scipy  :", _sp_check.__version__, "->", _sp_check.__file__)
# Kaggle's base image ships a stale torchao (0.10.0) that peft's LoRA module-dispatch chain
# unconditionally probes when wrapping ANY target module (even though we never use torchao/
# quantization) -- peft's is_torchao_available() raises rather than skipping on a too-old
# install. Purely an environment fix; does not enable or touch quantization.
pip("-U", "torchao>=0.16.0")
print("pip done")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 MB 32.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.67.0 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.67.0 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.

numpy  : 2.0.2 -> /usr/local/lib/python3.12/dist-packages/numpy/__init__.py
scipy  : 1.16.3 -> /usr/local/lib/python3.12/dist-packages/scipy/__init__.py
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 33.3 MB/s eta 0:00:00
pip done


In [3]:
import os, json, time, random, hashlib, base64
import numpy as np
# numpy/scipy private-API compat shim (same fix required in Experiment 4's Kaggle image)
import numpy._core._multiarray_umath as _mu
if not hasattr(_mu, "_blas_supports_fpe"):
    _mu._blas_supports_fpe = lambda *a, **k: False

# numpy.char / numpy.strings / numpy._core.strings / numpy._core.defchararray have shown THREE
# different, independent internal breakages across different Kaggle sessions with this SAME
# notebook code (missing compiled ufunc names; a missing __all__ on a half-initialized module;
# a real ufunc object that doesn't support __module__ assignment on this numpy build) -- each a
# different entry point, each only discovered after fixing the previous one. Rather than keep
# patching individual symptoms, unconditionally replace all four with inert placeholder modules
# BEFORE anything (torch/transformers/scipy/peft) can trigger any of them. We never use any
# numpy.char/numpy.strings functionality anywhere in this pipeline (audio, tokenization, JSON
# only), so short-circuiting them entirely -- regardless of whether this particular session's
# numpy happens to be broken -- is safe and removes this whole class of failure for good.
import sys, types


def _inert_numpy_string_module(name):
    m = types.ModuleType(name)
    m.__all__ = []
    m.__doc__ = f"Inert placeholder for {name} (unused in this pipeline)."
    return m


for _mod_name in ("numpy._core.strings", "numpy._core.defchararray", "numpy.char", "numpy.strings"):
    sys.modules[_mod_name] = _inert_numpy_string_module(_mod_name)
np.char = sys.modules["numpy.char"]
np.strings = sys.modules["numpy.strings"]
print("numpy.char/numpy.strings/numpy._core.strings/numpy._core.defchararray preemptively "
      "replaced with inert placeholders (never used in this pipeline) -- avoids this Kaggle "
      "image's numpy string-ufunc internals entirely, regardless of which entry point would "
      "otherwise trigger them.")
import torch

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

import transformers, peft
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("CUDA        :", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} = {torch.cuda.get_device_name(i)}, "
          f"{torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB")
DTYPE = torch.float16  # T4s -> fp16, matching every prior notebook in this project
os.makedirs(OUTPUT_DIR, exist_ok=True)


numpy.char/numpy.strings/numpy._core.strings/numpy._core.defchararray preemptively replaced with inert placeholders (never used in this pipeline) -- avoids this Kaggle image's numpy string-ufunc internals entirely, regardless of which entry point would otherwise trigger them.


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


torch       : 2.10.0+cu128
transformers: 5.17.0
peft        : 0.21.0
CUDA        : True | GPUs: 2
  cuda:0 = Tesla T4, 15.6 GB
  cuda:1 = Tesla T4, 15.6 GB


In [4]:
# ---- locate the Phase-3 train/val dataset ----
CANDIDATES = [
    "/kaggle/input/datasets/pranavjaiganesh13/phase3-train-val/kaggle_upload_phase3trainval", "/kaggle/input/phase3_train_val",
    "/kaggle/input/kaggle-upload-phase3trainval", "/kaggle/input/kaggle_upload_phase3trainval",
]
INPUT_DIR = None
for c in CANDIDATES:
    if os.path.exists(os.path.join(c, "phase3_train_nohist.json")):
        INPUT_DIR = c
        break
if INPUT_DIR is None:
    for base in ["/kaggle/input"]:
        if os.path.isdir(base):
            for name in os.listdir(base):
                p = os.path.join(base, name)
                if os.path.exists(os.path.join(p, "phase3_train_nohist.json")):
                    INPUT_DIR = p
                    break
assert INPUT_DIR is not None, (
    "Could not find the Phase-3 train/val dataset. Add the kaggle_upload_phase3trainval "
    "folder as a Kaggle Input dataset (Add Input), then re-run.")
AUDIO_DIR = os.path.join(INPUT_DIR, "audio")
assert os.path.isdir(AUDIO_DIR), f"audio/ not found under {INPUT_DIR}"

# Session-5 guard: this dataset must NEVER contain Session-5 files.
assert not os.path.exists(os.path.join(INPUT_DIR, "test.json")), (
    "test.json (Session 5) found in the input dataset -- STOP, wrong dataset attached")
print("INPUT_DIR:", INPUT_DIR)
print("AUDIO_DIR:", AUDIO_DIR, "|", len(os.listdir(AUDIO_DIR)), "wav files")


INPUT_DIR: /kaggle/input/datasets/pranavjaiganesh13/phase3-train-val/kaggle_upload_phase3trainval
AUDIO_DIR: /kaggle/input/datasets/pranavjaiganesh13/phase3-train-val/kaggle_upload_phase3trainval/audio | 5758 wav files


In [5]:
# ---- HF auth (optional; Qwen is ungated) ----
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF_TOKEN")
    if _tok:
        from huggingface_hub import login
        login(token=_tok)
        print("HF login OK")
except Exception as e:
    print("no HF_TOKEN secret / not needed for this ungated model:", repr(e))


HF login OK


In [6]:
!pip install -q scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 61.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.67.0 which is incompatible.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.1 which is incompatible.


In [7]:
# ================= iemocap_eval_lib.py, embedded verbatim as base64 =================
_EVAL_LIB_B64 = """..."""
_eval_lib_bytes = base64.b64decode(_EVAL_LIB_B64)
_actual_sha = hashlib.sha256(_eval_lib_bytes).hexdigest()
assert _actual_sha == EVAL_LIB_SHA256_EXPECTED, (
    f"iemocap_eval_lib.py sha256 mismatch: expected {EVAL_LIB_SHA256_EXPECTED}, got {_actual_sha}")
with open("/kaggle/working/iemocap_eval_lib.py", "wb") as f:
    f.write(_eval_lib_bytes)
sys.path.insert(0, "/kaggle/working")
import iemocap_eval_lib as EVAL
print("iemocap_eval_lib.py OK, sha256 =", _actual_sha[:16] + "...")
print("IEMOCAP_LABELS:", EVAL.IEMOCAP_LABELS)


iemocap_eval_lib.py OK, sha256 = 24fac1f49d6f049c...
IEMOCAP_LABELS: ['happy', 'sad', 'neutral', 'angry', 'excited', 'frustrated']


In [8]:
# ================= load train/val data for the selected ADAPTER_VARIANT =================
def _load(fn):
    with open(os.path.join(INPUT_DIR, fn), encoding="utf-8") as f:
        return json.load(f)

if SMOKE_MODE:
    train_records = _load(f"phase3_train_tiny_{ADAPTER_VARIANT}.json")
    val_records = _load(f"phase3_val_tiny_{ADAPTER_VARIANT}.json")
else:
    train_records = _load(f"phase3_train_{ADAPTER_VARIANT}.json")
    val_records = _load(f"phase3_val_{ADAPTER_VARIANT}.json")

HAS_HISTORY = (ADAPTER_VARIANT == "hist3")

# structural guards -- re-verified here, not just trusted from the local build
train_sessions = {r["session"] for r in train_records}
val_sessions = {r["session"] for r in val_records}
assert train_sessions <= {1, 2, 3}, f"train sessions must be subset of {{1,2,3}}, got {train_sessions}"
assert val_sessions == {4}, f"val sessions must be exactly {{4}}, got {val_sessions}"
train_vids = {r["video_id"] for r in train_records}
val_vids = {r["video_id"] for r in val_records}
assert not (train_vids & val_vids), "train/val dialogue overlap -- STOP"
if HAS_HISTORY:
    assert all("history_context" in r for r in train_records)
    assert all("history_context" in r for r in val_records)
else:
    assert all("history_context" not in r for r in train_records)
    assert all("history_context" not in r for r in val_records)

import collections
print(f"ADAPTER_VARIANT={ADAPTER_VARIANT}  SMOKE_MODE={SMOKE_MODE}")
print(f"train: {len(train_records)} rows | val: {len(val_records)} rows")
print("train per-class:", dict(collections.Counter(r["output"] for r in train_records)))
print("val   per-class:", dict(collections.Counter(r["output"] for r in val_records)))


ADAPTER_VARIANT=hist3  SMOKE_MODE=False
train: 4246 rows | val: 1512 rows
train per-class: {'neutral': 1066, 'frustrated': 987, 'angry': 606, 'sad': 696, 'happy': 387, 'excited': 504}
val   per-class: {'frustrated': 481, 'angry': 327, 'sad': 143, 'excited': 238, 'neutral': 258, 'happy': 65}


In [9]:
# ================= fresh-base-model loader + validated target-module derivation =================
# Re-derives TARGET_MODULES dynamically from the actual loaded model's named_modules(), using
# the EXACT collision-safe logic validated in kaggle_phase3_smoketest.ipynb (thinker.audio_tower
# vs thinker.model collision), rather than hardcoding the list -- so a protocol-drift assertion
# actually has something independent to check against, every time this function is called.
import re
from transformers import Qwen2_5OmniForConditionalGeneration as OmniCls
from peft import LoraConfig, get_peft_model, PeftModel

EXPECTED_N_TARGET_MODULES = 144
EXPECTED_N_TRAINABLE = 7_372_800

def derive_target_modules(base_model):
    proj_matches = [n for n, m in base_model.named_modules()
                    if isinstance(m, torch.nn.Linear) and re.search(r"\.(q|k|v|o)_proj$", n)]
    THINKER_LLM_PREFIX = "thinker.model."
    target_modules = sorted(n for n in proj_matches if n.startswith(THINKER_LLM_PREFIX))
    assert target_modules, "no thinker.model.* q/k/v/o_proj modules found -- architecture drift?"
    bad = [n for n in target_modules if any(seg in n for seg in
                                             ["audio_tower", "visual", "talker", "token2wav"])]
    assert not bad, f"target list still includes non-LLM pathway modules: {bad}"
    assert len(target_modules) == EXPECTED_N_TARGET_MODULES, (
        f"expected {EXPECTED_N_TARGET_MODULES} target modules, got {len(target_modules)} "
        f"-- architecture/checkpoint drift, STOP and re-run architecture inspection")
    return target_modules


def load_fresh_base_model():
    """Always loads a GENUINELY FRESH base model from the original checkpoint -- never
    reuses an already-PEFT-mutated object. Required both for a fresh start AND for resume
    (smoke-test Gate-6 lesson: wrapping an already-peft-tainted base object a second time
    produces an 'Already found a peft_config attribute' warning and an ambiguous state)."""
    m = OmniCls.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map="auto")
    return m


def apply_fresh_lora(base_model):
    """Fresh start: derive target modules, apply a NEW LoRA adapter, audit trainable params."""
    target_modules = derive_target_modules(base_model)
    lora_config = LoraConfig(task_type="CAUSAL_LM", r=LORA_R, lora_alpha=LORA_ALPHA,
                              lora_dropout=LORA_DROPOUT, target_modules=target_modules, bias="none")
    model = get_peft_model(base_model, lora_config)
    audit_trainable_params(model, target_modules)
    return model, target_modules


def load_lora_from_checkpoint(base_model, adapter_dir):
    """Resume: derive target modules (for the audit only -- the adapter's own config is what
    actually gets applied), load the SAVED adapter onto a fresh base model."""
    expected_target_modules = derive_target_modules(base_model)
    model = PeftModel.from_pretrained(base_model, adapter_dir, is_trainable=True)
    audit_trainable_params(model, expected_target_modules)
    return model, expected_target_modules


def audit_trainable_params(model, target_modules):
    """Preserve explicit assertions that ONLY the intended LoRA parameters are trainable --
    every time a model is constructed, fresh or resumed."""
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    # every trainable param name must contain "thinker.model."
    # and ".lora_"; every non-"thinker.model." param must be frozen.
    bad_trainable_outside_thinker = [n for n, p in model.named_parameters()
                                      if p.requires_grad and "thinker.model." not in n]
    frozen_violation = [n for n, p in model.named_parameters()
                        if (not p.requires_grad) and ("lora_" in n)]
    assert not bad_trainable_outside_thinker, (
        f"trainable params found outside thinker.model.*: {bad_trainable_outside_thinker[:5]}")
    assert not frozen_violation, f"LoRA params unexpectedly frozen: {frozen_violation[:5]}"
    assert n_trainable == EXPECTED_N_TRAINABLE, (
        f"expected exactly {EXPECTED_N_TRAINABLE} trainable params, got {n_trainable} -- STOP")
    audio_vision_talker_trainable = [n for n, p in model.named_parameters()
                                     if p.requires_grad and any(seg in n for seg in
                                     ["audio_tower", "visual", "talker", "token2wav"])]
    assert not audio_vision_talker_trainable, (
        f"audio/vision/talker params unexpectedly trainable: {audio_vision_talker_trainable[:5]}")
    not_lora = [tm for tm in target_modules
                if not (hasattr(model.get_submodule(tm), "lora_A") and
                        hasattr(model.get_submodule(tm), "lora_B"))]
    assert not not_lora, f"target modules missing lora_A/lora_B: {not_lora[:5]}"
    print(f"  trainable params: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.4f}%)  "
          f"[EXPECTED {EXPECTED_N_TRAINABLE:,}] -- OK")
    print(f"  zero trainable params under audio/vision/talker: CONFIRMED")
    return n_trainable, n_total


In [10]:
# ================= processor + prompt/example construction (validated pattern, reused) =================
from transformers import AutoProcessor
proc = AutoProcessor.from_pretrained(MODEL_ID)

QWEN_SYS = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
            "capable of perceiving auditory and visual inputs, as well as generating text and speech.")


def _render_prompt_and_mm(rec, history_context):
    prompt_text = EVAL.build_prompt(rec["utterance"], history_context)
    wav_path = os.path.join(AUDIO_DIR, rec["id"] + ".wav")
    conv = [
        {"role": "system", "content": [{"type": "text", "text": QWEN_SYS}]},
        {"role": "user", "content": [
            {"type": "audio", "audio": wav_path},
            {"type": "text", "text": prompt_text}]},
    ]
    from qwen_omni_utils import process_mm_info
    rendered = proc.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
    try:
        audios, images, videos = process_mm_info(conv, use_audio_in_video=False)
    except TypeError:
        audios, images, videos = process_mm_info(conv)
    try:
        prompt_inputs = proc(text=rendered, audio=audios, images=images, videos=videos,
                             return_tensors="pt", padding=False)
    except TypeError:
        prompt_inputs = proc(text=rendered, audios=audios, images=images, videos=videos,
                             return_tensors="pt", padding=False)
    return prompt_inputs


def build_training_inputs(rec, history_context):
    """ONE training example: completion-only label masking (labels=[-100]*len(prompt)+answer_ids),
    real raw audio conditioning (Diagnostic E-validated path) -- identical construction to the
    Phase-3 smoke test's build_training_inputs()."""
    prompt_inputs = _render_prompt_and_mm(rec, history_context)
    prompt_ids = prompt_inputs["input_ids"][0]
    answer_ids = proc.tokenizer.encode(" " + rec["output"], add_special_tokens=False)
    eos_id = proc.tokenizer.eos_token_id
    answer_ids = torch.tensor(answer_ids + [eos_id], dtype=prompt_ids.dtype)

    full_ids = torch.cat([prompt_ids, answer_ids])
    labels = torch.cat([torch.full_like(prompt_ids, -100), answer_ids])
    attn = torch.ones_like(full_ids)

    out = dict(prompt_inputs)
    out["input_ids"] = full_ids.unsqueeze(0)
    out["attention_mask"] = attn.unsqueeze(0)
    out["labels"] = labels.unsqueeze(0)
    return out


def build_generation_inputs(rec, history_context):
    """Prompt-only (no answer appended), for model.generate() at validation time."""
    return _render_prompt_and_mm(rec, history_context)


def to_model_device(inputs, ref_model):
    # BatchFeature.to()'s "cast only floating tensors to model dtype, leave integer tensors
    # alone" behavior, replicated manually since build_training_inputs() returns a plain dict.
    first_param = next(ref_model.parameters())
    device, dtype = first_param.device, first_param.dtype
    moved = {}
    for k, v in inputs.items():
        if torch.is_tensor(v):
            v = v.to(device)
            if torch.is_floating_point(v):
                v = v.to(dtype)
            moved[k] = v
        else:
            moved[k] = v
    return moved


# sanity print for one example (no model call yet)
_ex = build_training_inputs(train_records[0], train_records[0].get("history_context"))
print("built one training example. input_ids shape:", _ex["input_ids"].shape,
      "labels shape:", _ex["labels"].shape,
      "n_masked(-100):", int((_ex["labels"][0] == -100).sum()),
      "n_supervised:", int((_ex["labels"][0] != -100).sum()))


preprocessor_config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: audio_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

built one training example. input_ids shape: torch.Size([1, 171]) labels shape: torch.Size([1, 171]) n_masked(-100): 169 n_supervised: 2


In [11]:
# ================= checkpoint / resume helpers =================
# "Do not wrap an already PEFT-mutated base object" (smoke-test Gate-6 lesson): every path
# below (fresh start AND resume) calls load_fresh_base_model() to get a clean object first.
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "adapter")
OPT_PATH = os.path.join(OUTPUT_DIR, "optimizer.pt")
STATE_PATH = os.path.join(OUTPUT_DIR, "train_state.json")
LOG_PATH = os.path.join(OUTPUT_DIR, "training_log.json")


def find_resume_dir():
    """Same-session resume: OUTPUT_DIR already has a checkpoint from an earlier run of THIS
    cell. Cross-session resume: RESUME_INPUT_DIR points at a previous COMMITTED run's output,
    added as a Kaggle Input dataset (read-only -- copied into OUTPUT_DIR before use)."""
    if os.path.exists(STATE_PATH) and os.path.isdir(ADAPTER_DIR):
        return OUTPUT_DIR
    if RESUME_INPUT_DIR and os.path.exists(os.path.join(RESUME_INPUT_DIR, "train_state.json")):
        return RESUME_INPUT_DIR
    return None


def save_checkpoint(model, optimizer, state, tag="latest"):
    save_dir = ADAPTER_DIR if tag == "latest" else os.path.join(OUTPUT_DIR, f"adapter_{tag}")
    tmp_dir = save_dir + ".tmp"
    model.save_pretrained(tmp_dir)
    if os.path.isdir(save_dir):
        import shutil
        shutil.rmtree(save_dir)
    os.replace(tmp_dir, save_dir)
    if tag == "latest":
        torch.save(optimizer.state_dict(), OPT_PATH + ".tmp")
        os.replace(OPT_PATH + ".tmp", OPT_PATH)
        with open(STATE_PATH + ".tmp", "w", encoding="utf-8") as f:
            json.dump(state, f, indent=2)
        os.replace(STATE_PATH + ".tmp", STATE_PATH)


def append_log(entry):
    log = []
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH, encoding="utf-8") as f:
            log = json.load(f)
    log.append(entry)
    with open(LOG_PATH + ".tmp", "w", encoding="utf-8") as f:
        json.dump(log, f, indent=2)
    os.replace(LOG_PATH + ".tmp", LOG_PATH)
    return log


def enable_gradient_checkpointing(model, thinker):
    thinker.gradient_checkpointing_enable()
    thinker.config.use_cache = False
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()
    else:
        def _make_inputs_require_grad(module, inp, out):
            out.requires_grad_(True)
        thinker.get_input_embeddings().register_forward_hook(_make_inputs_require_grad)
    print("gradient checkpointing enabled on model.thinker; use_cache=False during training")


In [12]:
# ================= validation: full matched-format Session-4 generation + scoring =================
def run_validation(model, thinker, val_records, epoch_tag):
    """Matched-format validation: this adapter's own training format (NoHist or Hist3) is
    the SAME format used here -- 'one complete matched Session-4 validation per epoch'.
    Inference uses the top-level model.generate(...) (validated, working path), NOT
    thinker(**batch) (that path is for training loss only)."""
    thinker.config.use_cache = True
    model.eval()
    raw_answers = []
    t0 = time.time()
    with torch.no_grad():
        for i, rec in enumerate(val_records):
            history_context = rec.get("history_context")
            inputs = build_generation_inputs(rec, history_context)
            inputs = to_model_device(inputs, ref_model=model)
            try:
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                     return_audio=False)
            except TypeError:
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
            if isinstance(out, (tuple, list)):
                out = out[0]
            gen = out[:, inputs["input_ids"].shape[1]:]
            raw = proc.batch_decode(gen, skip_special_tokens=True)[0].strip()
            raw_answers.append(raw)
            if (i + 1) % 200 == 0:
                elapsed = time.time() - t0
                print(f"    validation {i+1}/{len(val_records)}  ({elapsed:.0f}s elapsed, "
                      f"{elapsed/(i+1):.2f}s/example)")
    thinker.config.use_cache = False
    model.train()

    metrics = EVAL.score_predictions_normalized(val_records, raw_answers)

    preds_path = os.path.join(OUTPUT_DIR, f"val_predictions_{epoch_tag}.json")
    with open(preds_path + ".tmp", "w", encoding="utf-8") as f:
        json.dump(metrics["predictions"], f, ensure_ascii=False, indent=2)
    os.replace(preds_path + ".tmp", preds_path)
    metrics_path = os.path.join(OUTPUT_DIR, f"val_metrics_{epoch_tag}.json")
    with open(metrics_path + ".tmp", "w", encoding="utf-8") as f:
        json.dump({k: v for k, v in metrics.items() if k not in ("predictions",)},
                   f, ensure_ascii=False, indent=2, default=str)
    os.replace(metrics_path + ".tmp", metrics_path)

    print(f"  VAL[{epoch_tag}]: WA={metrics['accuracy_WA']}  UA={metrics['UA_unweighted_recall']}  "
          f"macro_f1={metrics['macro_f1']}  weighted_f1={metrics['weighted_f1']}  "
          f"match_tiers={metrics['match_tier_counts']}")
    return metrics


In [13]:
# ================= RNG state (de)serialization, for as-deterministic-as-possible resume =================
def rng_state_to_json():
    np_state = np.random.get_state()
    py_state = random.getstate()
    out = {
        "torch": torch.get_rng_state().tolist(),
        "cuda": [t.tolist() for t in torch.cuda.get_rng_state_all()] if torch.cuda.is_available() else [],
        "numpy": [np_state[0], np_state[1].tolist(), int(np_state[2]), int(np_state[3]), float(np_state[4])],
        "python": [py_state[0], list(py_state[1]), py_state[2]],
    }
    return out


def rng_state_from_json(d):
    torch.set_rng_state(torch.tensor(d["torch"], dtype=torch.uint8))
    if torch.cuda.is_available() and d.get("cuda"):
        torch.cuda.set_rng_state_all([torch.tensor(t, dtype=torch.uint8) for t in d["cuda"]])
    npd = d["numpy"]
    np.random.set_state((npd[0], np.array(npd[1], dtype=np.uint32), npd[2], npd[3], npd[4]))
    pyd = d["python"]
    random.setstate((pyd[0], tuple(pyd[1]), pyd[2]))


def deterministic_epoch_perm(n, epoch):
    perm = list(range(n))
    random.Random(SEED * 1000 + epoch).shuffle(perm)
    return perm


In [14]:
# ================= main training loop =================
def train_adapter(train_records, val_records, max_epochs, patience, tag):
    resume_dir = find_resume_dir()
    base_model = load_fresh_base_model()

    if resume_dir:
        adapter_src = os.path.join(resume_dir, "adapter")
        model, target_modules = load_lora_from_checkpoint(base_model, adapter_src)
        with open(os.path.join(resume_dir, "train_state.json"), encoding="utf-8") as f:
            state = json.load(f)
        opt_src = os.path.join(resume_dir, "optimizer.pt")
        print(f"RESUMING from {resume_dir}: next_epoch={state['next_epoch']}, "
              f"examples_consumed_in_epoch={state['examples_consumed_in_epoch']}, "
              f"global_step={state['global_step']}, best_macro_f1={state['best_macro_f1']}")
    else:
        model, target_modules = apply_fresh_lora(base_model)
        state = {"next_epoch": 1, "examples_consumed_in_epoch": 0, "global_step": 0,
                 "best_macro_f1": -1.0, "best_epoch": None, "patience_counter": 0}
        opt_src = None
        print("FRESH START")

    thinker = model.thinker
    enable_gradient_checkpointing(model, thinker)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=ADAMW_LR, betas=ADAMW_BETAS)
    if opt_src and os.path.exists(opt_src):
        optimizer.load_state_dict(torch.load(opt_src, map_location="cpu"))
        print(f"  optimizer state restored from {opt_src}")

    if resume_dir and "rng_state" in state:
        try:
            rng_state_from_json(state["rng_state"])
            print("  RNG state restored (torch/cuda/numpy/python)")
        except Exception as e:
            print("  WARNING: could not fully restore RNG state:", repr(e))

    model.train()
    thinker.config.use_cache = False

    first_step_diagnostic_done = False

    def do_first_step_diagnostic():
        # "use a trainable tensor known to have a nonzero gradient (normally an appropriate
        # lora_B after backward), so the observed change is gradient-driven rather than merely
        # AdamW weight decay" -- smoke-test Gate-5 lesson, fixed here.
        probe_name, probe_param = next(
            (n, p) for n, p in model.named_parameters()
            if p.requires_grad and n.endswith("lora_B.default.weight"))
        assert probe_param.grad is not None and probe_param.grad.abs().sum().item() > 0, (
            f"expected probe param {probe_name} (a lora_B) to have a nonzero gradient on the "
            f"first optimizer step -- if this fails, the gradient-driven-update check below "
            f"would be meaningless")
        before = probe_param.detach().clone()
        return probe_name, probe_param, before

    def check_first_step_diagnostic(probe_name, probe_param, before):
        after = probe_param.detach().clone()
        changed = not torch.equal(before, after)
        print(f"  [first-step diagnostic] probe={probe_name} (lora_B, known nonzero grad)  "
              f"changed_after_step={changed}  max_abs_delta={float((after-before).abs().max()):.6g}")
        assert changed, "optimizer step did not change a parameter with a known nonzero gradient"

    for epoch in range(state["next_epoch"], max_epochs + 1):
        perm = deterministic_epoch_perm(len(train_records), epoch)
        start_idx = state["examples_consumed_in_epoch"] if epoch == state["next_epoch"] else 0
        print(f"\n{'='*78}\nEPOCH {epoch}/{max_epochs}  (resuming at example {start_idx}/{len(perm)})\n{'='*78}")

        idx_stream = perm[start_idx:]
        epoch_losses = []
        i = 0
        while i < len(idx_stream):
            microbatch_idxs = idx_stream[i:i + GRAD_ACCUM]
            optimizer.zero_grad(set_to_none=True)
            step_losses = []
            for ridx in microbatch_idxs:
                rec = train_records[ridx]
                history_context = rec.get("history_context")
                ex = build_training_inputs(rec, history_context)
                batch = to_model_device(ex, ref_model=thinker)
                out = thinker(**batch)
                loss = out.loss / len(microbatch_idxs)
                loss.backward()
                step_losses.append(loss.item() * len(microbatch_idxs))
            if not first_step_diagnostic_done:
                probe = do_first_step_diagnostic()
            optimizer.step()
            if not first_step_diagnostic_done:
                check_first_step_diagnostic(*probe)
                first_step_diagnostic_done = True
            i += len(microbatch_idxs)
            state["global_step"] += 1
            state["examples_consumed_in_epoch"] = start_idx + i
            epoch_losses.extend(step_losses)

            if state["global_step"] % SAVE_EVERY_N_OPT_STEPS == 0:
                state["next_epoch"] = epoch
                state["rng_state"] = rng_state_to_json()
                save_checkpoint(model, optimizer, state, tag="latest")
                print(f"  [checkpoint] global_step={state['global_step']}  "
                      f"examples_consumed_in_epoch={state['examples_consumed_in_epoch']}/{len(perm)}")

        # ---- end of epoch: full matched-format validation ----
        mean_train_loss = sum(epoch_losses) / max(1, len(epoch_losses))
        val_metrics = run_validation(model, thinker, val_records, epoch_tag=f"{tag}_epoch{epoch}")
        macro_f1 = val_metrics["macro_f1"]
        macro_f1_rounded = round(macro_f1, MACRO_F1_ROUND_NDIGITS)
        best_rounded = round(state["best_macro_f1"], MACRO_F1_ROUND_NDIGITS)
        improved = macro_f1_rounded > best_rounded   # strict >: a rounded TIE does NOT
                                                       # replace the earlier-epoch best
        if improved:
            state["best_macro_f1"] = macro_f1
            state["best_epoch"] = epoch
            state["patience_counter"] = 0
            save_checkpoint(model, optimizer, state, tag="best")
            print(f"  NEW BEST: macro_f1={macro_f1} at epoch {epoch}")
        else:
            state["patience_counter"] += 1
            print(f"  no improvement (rounded macro_f1={macro_f1_rounded} vs best="
                  f"{best_rounded} at epoch {state['best_epoch']}); "
                  f"patience_counter={state['patience_counter']}/{patience}")

        append_log({
            "epoch": epoch, "global_step": state["global_step"],
            "train_loss_mean": mean_train_loss,
            "learning_rate": ADAMW_LR,
            "val_accuracy_WA": val_metrics["accuracy_WA"],
            "val_UA": val_metrics["UA_unweighted_recall"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "match_tier_counts": val_metrics["match_tier_counts"],
            "n_ambiguous_multi_label": val_metrics["n_ambiguous_multi_label"],
            "best_macro_f1_so_far": state["best_macro_f1"],
            "best_epoch_so_far": state["best_epoch"],
            "patience_counter": state["patience_counter"],
            "improved_this_epoch": improved,
        })

        state["next_epoch"] = epoch + 1
        state["examples_consumed_in_epoch"] = 0
        state["rng_state"] = rng_state_to_json()
        save_checkpoint(model, optimizer, state, tag="latest")

        if state["patience_counter"] >= patience:
            print(f"\nEARLY STOPPING at epoch {epoch} (patience={patience} exceeded). "
                  f"Best macro_f1={state['best_macro_f1']} at epoch {state['best_epoch']}.")
            break

    print(f"\nDONE. best_epoch={state['best_epoch']}  best_macro_f1={state['best_macro_f1']}")
    return model, state


In [15]:
# ================= RUN =================
max_epochs = MAX_EPOCHS_SMOKE if SMOKE_MODE else MAX_EPOCHS_FULL
patience = PATIENCE_SMOKE if SMOKE_MODE else PATIENCE_FULL
print(f"Starting {'SMOKE' if SMOKE_MODE else 'FULL'} training run: "
      f"variant={ADAPTER_VARIANT}  max_epochs={max_epochs}  patience={patience}  "
      f"train_n={len(train_records)}  val_n={len(val_records)}  output_dir={OUTPUT_DIR}")

model, final_state = train_adapter(train_records, val_records, max_epochs, patience, tag=RUN_TAG)

print("\n" + "="*78)
print("RUN COMPLETE" if not SMOKE_MODE else "SMOKE RUN COMPLETE -- verify the checks above, "
      "then set SMOKE_MODE=False and Restart & Run All for the real training run.")
print("="*78)
print(json.dumps(final_state, indent=2, default=str))


Starting FULL training run: variant=hist3  max_epochs=8  patience=3  train_n=4246  val_n=1512  output_dir=/kaggle/working/phase3_train_hist3_full


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2543 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-3B
Key                                                | Status     |  | 
---------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.rotary_embed.inv_freq | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

spk_dict.pt:   0%|          | 0.00/260k [00:00<?, ?B/s]

  trainable params: 7,372,800 / 5,544,493,440 (0.1330%)  [EXPECTED 7,372,800] -- OK
  zero trainable params under audio/vision/talker: CONFIRMED
FRESH START
gradient checkpointing enabled on model.thinker; use_cache=False during training

EPOCH 1/8  (resuming at example 0/4246)


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


  [first-step diagnostic] probe=base_model.model.thinker.model.layers.0.self_attn.q_proj.lora_B.default.weight (lora_B, known nonzero grad)  changed_after_step=True  max_abs_delta=9.99997e-05
  [checkpoint] global_step=100  examples_consumed_in_epoch=800/4246
  [checkpoint] global_step=200  examples_consumed_in_epoch=1600/4246
  [checkpoint] global_step=300  examples_consumed_in_epoch=2400/4246
  [checkpoint] global_step=400  examples_consumed_in_epoch=3200/4246
  [checkpoint] global_step=500  examples_consumed_in_epoch=4000/4246
    validation 200/1512  (103s elapsed, 0.52s/example)
    validation 400/1512  (207s elapsed, 0.52s/example)
    validation 600/1512  (311s elapsed, 0.52s/example)
    validation 800/1512  (414s elapsed, 0.52s/example)
    validation 1000/1512  (522s elapsed, 0.52s/example)
    validation 1200/1512  (626s elapsed, 0.52s/example)
    validation 1400/1512  (733s elapsed, 0.52s/example)
  VAL[hist3_full_epoch1]: WA=64.418  UA=56.348  macro_f1=57.926  weighted_f1

## Kaggle instructions

1. **Add Input**: the `kaggle_upload_phase3trainval` dataset (built by
   `kaggle_prep/export_phase3_train_val_audio.py`). For a cross-session resume, also add
   your own previous COMMITTED run's output as a second Input dataset, and set
   `RESUME_INPUT_DIR` in the CONFIG cell to its `/kaggle/input/<name>` path.
2. **Settings**: Accelerator = GPU T4 x2, Internet = On.
3. Set `ADAPTER_VARIANT` ("nohist" or "hist3") and `SMOKE_MODE` in the CONFIG cell.
4. **First run**: `SMOKE_MODE = True`, Run All. Confirm the first-step diagnostic shows
   `changed_after_step=True` on a `lora_B` probe with a genuinely nonzero pre-step gradient,
   validation runs and prints a macro_f1 number, and a checkpoint is written under
   `/kaggle/working/phase3_train_<variant>_smoke/`.
5. **Real run**: set `SMOKE_MODE = False`, Restart the session (clears any smoke-mode
   in-memory state), Run All. This is `/kaggle/working/phase3_train_<variant>_full/` --
   a separate directory from the smoke run, never mixed.
6. If the session disconnects or times out mid-run: reopen the notebook and Run All again --
   the training loop finds the existing checkpoint in `OUTPUT_DIR` and resumes automatically
   (same session). If starting a genuinely NEW Kaggle session (new browser tab after closing,
   or the working directory was reset): first **Save Version -> Save & Run All (Commit)** on
   the run you want to preserve, then start a new session with that commit's output added as
   an Input dataset, and set `RESUME_INPUT_DIR` to point at it.
7. Repeat steps 3-6 with `ADAPTER_VARIANT = "hist3"` as a **separate** Kaggle
   session/notebook copy -- do not run both variants in the same session or output directory.
8. Do not add the Session-5 dataset to this notebook at any point.
